In [ ]:
!pip install psycopg2-binary sqlalchemy -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from sqlalchemy import create_engine, text
import os

print("✅ Libraries loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 16.8 MB/s eta 0:00:00
Mounted at /content/drive
✅ Libraries loaded


In [ ]:
folder = "/content/drive/MyDrive/P1_Airline"

FILES = [
    "bts_ontime_2024.csv",
    "bts_ontime_2025.csv",
    "bts_ontime_2026.csv"
]

COLS = [
    'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest', 'DestCityName', 'DestState',
    'DepDelay', 'DepDelayMinutes', 'ArrDelay', 'ArrDelayMinutes',
    'Cancelled', 'CancellationCode', 'Diverted',
    'AirTime', 'Distance',
    'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay'
]

CHUNK_SIZE = 100_000

print("✅ Config ready")

✅ Config ready


In [ ]:
create_tables_sql = """
DROP TABLE IF EXISTS delays CASCADE;
DROP TABLE IF EXISTS flights CASCADE;
DROP TABLE IF EXISTS airports CASCADE;
DROP TABLE IF EXISTS carriers CASCADE;

CREATE TABLE carriers (
    carrier_code    VARCHAR(10) PRIMARY KEY,
    carrier_name    VARCHAR(100)
);

CREATE TABLE airports (
    airport_code    VARCHAR(10) PRIMARY KEY,
    city            VARCHAR(100),
    state           VARCHAR(50)
);

CREATE TABLE flights (
    flight_id           SERIAL PRIMARY KEY,
    carrier_code        VARCHAR(10) REFERENCES carriers(carrier_code),
    flight_number       INT,
    flight_date         DATE,
    origin              VARCHAR(10) REFERENCES airports(airport_code),
    dest                VARCHAR(10) REFERENCES airports(airport_code),
    dep_delay_min       FLOAT,
    arr_delay_min       FLOAT,
    cancelled           SMALLINT,
    cancellation_code   VARCHAR(5),
    diverted            SMALLINT,
    air_time            FLOAT,
    distance            FLOAT
);

CREATE TABLE delays (
    delay_id            SERIAL PRIMARY KEY,
    flight_id           INT REFERENCES flights(flight_id),
    carrier_delay       FLOAT,
    weather_delay       FLOAT,
    nas_delay           FLOAT,
    security_delay      FLOAT,
    late_aircraft_delay FLOAT
);
"""

with engine.connect() as conn:
    conn.execute(text(create_tables_sql))
    conn.commit()

print("✅ All 4 tables created in Neon PostgreSQL")

✅ All 4 tables created in Neon PostgreSQL


In [ ]:
print("🚀 Starting ETL pipeline (smart sample)...\n")

airports_seen = set()
carriers_seen = set()
flight_id_counter = 1

# Strategy: load only 2025 data, sample 50k rows per month
# This gives us 600k rows max — fits in 512MB comfortably
filepath = f"{folder}/bts_ontime_2025.csv"
print(f"📂 Processing bts_ontime_2025.csv...")

monthly_chunks = []
chunk_rows = 0

for chunk in pd.read_csv(filepath, usecols=COLS, chunksize=CHUNK_SIZE, low_memory=False):

    # --- CLEAN ---
    chunk.rename(columns={
        'Reporting_Airline': 'carrier_code',
        'Flight_Number_Reporting_Airline': 'flight_number',
        'FlightDate': 'flight_date',
        'Origin': 'origin', 'OriginCityName': 'origin_city', 'OriginState': 'origin_state',
        'Dest': 'dest', 'DestCityName': 'dest_city', 'DestState': 'dest_state',
        'DepDelay': 'dep_delay', 'DepDelayMinutes': 'dep_delay_min',
        'ArrDelay': 'arr_delay', 'ArrDelayMinutes': 'arr_delay_min',
        'Cancelled': 'cancelled', 'CancellationCode': 'cancellation_code',
        'Diverted': 'diverted', 'AirTime': 'air_time', 'Distance': 'distance',
        'CarrierDelay': 'carrier_delay', 'WeatherDelay': 'weather_delay',
        'NASDelay': 'nas_delay', 'SecurityDelay': 'security_delay',
        'LateAircraftDelay': 'late_aircraft_delay'
    }, inplace=True)

    chunk['flight_date'] = pd.to_datetime(chunk['flight_date'])
    chunk.dropna(subset=['origin', 'dest', 'carrier_code'], inplace=True)

    delay_cols = ['dep_delay', 'dep_delay_min', 'arr_delay', 'arr_delay_min',
                  'carrier_delay', 'weather_delay', 'nas_delay',
                  'security_delay', 'late_aircraft_delay']
    chunk[delay_cols] = chunk[delay_cols].fillna(0)
    chunk['cancelled'] = chunk['cancelled'].fillna(0).astype(int)
    chunk['diverted'] = chunk['diverted'].fillna(0).astype(int)
    chunk['cancellation_code'] = chunk['cancellation_code'].fillna('N')

    monthly_chunks.append(chunk)
    chunk_rows += len(chunk)

# Combine all chunks
print(f"   Total rows in 2025 file: {chunk_rows:,}")
df_full = pd.concat(monthly_chunks, ignore_index=True)
del monthly_chunks

# Smart sample: 40,000 rows per month = ~480,000 total rows
df_full['month'] = df_full['flight_date'].dt.month
df_sampled = df_full.groupby('month', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 40000), random_state=42)
).reset_index(drop=True)
del df_full

print(f"   Sampled rows: {len(df_sampled):,} (balanced across all 12 months)")

# --- AIRPORTS ---
origins = df_sampled[['origin','origin_city','origin_state']].rename(
    columns={'origin':'airport_code','origin_city':'city','origin_state':'state'})
dests = df_sampled[['dest','dest_city','dest_state']].rename(
    columns={'dest':'airport_code','dest_city':'city','dest_state':'state'})
airports_df = pd.concat([origins, dests]).drop_duplicates(subset='airport_code').reset_index(drop=True)
airports_df.to_sql('airports', engine, if_exists='append', index=False)
print(f"   ✅ Airports: {len(airports_df):,} unique airports")

# --- CARRIERS ---
carriers_df = df_sampled[['carrier_code']].drop_duplicates().reset_index(drop=True)
carriers_df['carrier_name'] = carriers_df['carrier_code']
carriers_df.to_sql('carriers', engine, if_exists='append', index=False)
print(f"   ✅ Carriers: {len(carriers_df):,} unique carriers")

# --- FLIGHTS ---
df_sampled.insert(0, 'flight_id', range(1, len(df_sampled) + 1))
flights_df = df_sampled[[
    'flight_id', 'carrier_code', 'flight_number', 'flight_date',
    'origin', 'dest', 'dep_delay_min', 'arr_delay_min',
    'cancelled', 'cancellation_code', 'diverted', 'air_time', 'distance'
]].copy()
flights_df.to_sql('flights', engine, if_exists='append', index=False, chunksize=10_000)
print(f"   ✅ Flights: {len(flights_df):,} rows")

# --- DELAYS ---
delays_df = df_sampled[[
    'flight_id', 'carrier_delay', 'weather_delay',
    'nas_delay', 'security_delay', 'late_aircraft_delay'
]].copy()
delays_df.to_sql('delays', engine, if_exists='append', index=False, chunksize=10_000)
print(f"   ✅ Delays: {len(delays_df):,} rows")

print("\n🎉 ETL Complete!")

🚀 Starting ETL pipeline (smart sample)...

📂 Processing bts_ontime_2025.csv...
   Total rows in 2025 file: 7,001,619


/tmp/ipykernel_2350/1195127474.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled = df_full.groupby('month', group_keys=False).apply(


   Sampled rows: 480,000 (balanced across all 12 months)
   ✅ Airports: 351 unique airports
   ✅ Carriers: 14 unique carriers
   ✅ Flights: 480,000 rows
   ✅ Delays: 480,000 rows

🎉 ETL Complete!


In [ ]:
print("🔍 Data Quality Report\n")

checks = {
    "Total flights":      "SELECT COUNT(*) FROM flights",
    "Cancelled flights":  "SELECT COUNT(*) FROM flights WHERE cancelled = 1",
    "Diverted flights":   "SELECT COUNT(*) FROM flights WHERE diverted = 1",
    "Unique carriers":    "SELECT COUNT(*) FROM carriers",
    "Unique airports":    "SELECT COUNT(*) FROM airports",
    "Avg arrival delay":  "SELECT ROUND(AVG(arr_delay_min)::numeric, 2) FROM flights",
    "Max arrival delay":  "SELECT MAX(arr_delay_min) FROM flights",
    "Total delay rows":   "SELECT COUNT(*) FROM delays",
    "Date range":         "SELECT MIN(flight_date)::text || ' → ' || MAX(flight_date)::text FROM flights",
}

with engine.connect() as conn:
    for label, query in checks.items():
        result = conn.execute(text(query)).fetchone()[0]
        print(f"  {label:<25} {str(result):>20}")

print("\n✅ Database is clean and ready for SQL queries!")

🔍 Data Quality Report

  Total flights                           480000
  Cancelled flights                         7124
  Diverted flights                          1317
  Unique carriers                             14
  Unique airports                            351
  Avg arrival delay                        16.81
  Max arrival delay                       3275.0
  Total delay rows                        480000
  Date range                2025-01-01 → 2025-12-31

✅ Database is clean and ready for SQL queries!


In [ ]:
# use this because of error. if you not get any error then don't run this.
with engine.connect() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS delays CASCADE;
        DROP TABLE IF EXISTS flights CASCADE;
        DROP TABLE IF EXISTS airports CASCADE;
        DROP TABLE IF EXISTS carriers CASCADE;
    """))
    conn.commit()
print("✅ Database cleared")

✅ Database cleared
